### Data Preprocessing

#### Objective

The objective of this notebook is to clean, transform, and prepare the Netflix dataset for exploratory data analysis.

This phase involves identifying and handling data quality issues, converting data into appropriate formats, engineering new features, and validating the processed dataset to ensure consistency and reliability for subsequent analysis.

The original dataset will remain unchanged, while all transformations will be applied to a working copy of the data.

In [37]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv("../data/raw/Netflix Dataset.csv")

In [7]:
netflix = df.copy()

In [8]:
netflix.info()

<class 'pandas.DataFrame'>
RangeIndex: 7789 entries, 0 to 7788
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Show_Id       7789 non-null   str  
 1   Category      7789 non-null   str  
 2   Title         7789 non-null   str  
 3   Director      5401 non-null   str  
 4   Cast          7071 non-null   str  
 5   Country       7282 non-null   str  
 6   Release_Date  7779 non-null   str  
 7   Rating        7782 non-null   str  
 8   Duration      7789 non-null   str  
 9   Type          7789 non-null   str  
 10  Description   7789 non-null   str  
dtypes: str(11)
memory usage: 669.5 KB


In [9]:
# Missing values
def missing_values(df):

    missing = pd.DataFrame({
        "Missing Values": df.isnull().sum(),
        "Percentage": (df.isnull().sum()/len(df))*100
    })

    return missing.sort_values("Missing Values", ascending=False)

In [10]:
#Duplicate check
def duplicate_summary(df):

    duplicates = df.duplicated().sum()

    print(f"Duplicate Rows: {duplicates}")

In [11]:
missing_values(netflix)

,Missing Values,Percentage
Director,2388,30.658621
Cast,718,9.218128
Country,507,6.509180
Release_Date,10,0.128386
Rating,7,0.089870
Show_Id,0,0.000000
Category,0,0.000000
Title,0,0.000000
Duration,0,0.000000
Type,0,0.000000


In [12]:
# This provides only the details of the columns where there are missing values
missing_df = pd.DataFrame({
    "Missing Values": netflix.isnull().sum(),
    "Percentage (%)": round((netflix.isnull().sum() / len(netflix)) * 100, 2)
})

missing_df = missing_df[missing_df["Missing Values"] > 0]
missing_df.sort_values(by="Missing Values", ascending=False)

,Missing Values,Percentage (%)
Director,2388,30.66
Cast,718,9.22
Country,507,6.51
Release_Date,10,0.13
Rating,7,0.09


| Column       | Action                   | Reason                                                           |
| ------------ | ------------------------ | ---------------------------------------------------------------- |
| Director     | Replace with `"Unknown"` | Missing director information does not invalidate the record.     |
| Cast         | Replace with `"Unknown"` | Many documentaries and specials lack cast information.           |
| Country      | Replace with `"Unknown"` | Retains records for later analysis without dropping data.        |
| Rating       | Fill with mode           | Very few missing values.                                         |
| Release_Date | Drop rows                | Only 10 missing records; dates are important for trend analysis. |


In [15]:
# Handling the director column
netflix["Director"].isnull().sum()
netflix["Director"] = netflix["Director"].fillna("Unknown") #Replacing the null values with "Unknow"
netflix["Director"].isnull().sum() # Showing the total missing values after replacement, expected to be 0

np.int64(0)

In [16]:
#Handling cast column
netflix["Cast"] = netflix["Cast"].fillna("Unknown")
netflix["Cast"].isnull().sum()

np.int64(0)

In [17]:
# Country column
netflix["Country"] = netflix["Country"].fillna("Unknown")
netflix["Country"].isnull().sum()

np.int64(0)

In [21]:
#Rating
mode_rating = netflix["Rating"].mode()[0]
netflix["Rating"] = netflix["Rating"].fillna(mode_rating)
netflix["Rating"].isnull().sum()

np.int64(0)

In [24]:
#Release Date
print(f"Rows before: {netflix.shape[0]}")
netflix = netflix.dropna(subset=["Release_Date"]) # Dropping the rows with missing dates, expected to drop 10 rows
print(f"Rows after: {netflix.shape[0]}")

Rows before: 7779
Rows after: 7779


In [25]:
missing_df = pd.DataFrame({
    "Missing Values": netflix.isnull().sum(),
    "Percentage (%)": round((netflix.isnull().sum() / len(netflix)) * 100, 2)
})

missing_df

,Missing Values,Percentage (%)
Show_Id,0,0.0
Category,0,0.0
Title,0,0.0
Director,0,0.0
Cast,0,0.0
Country,0,0.0
Release_Date,0,0.0
Rating,0,0.0
Duration,0,0.0
Type,0,0.0


In [26]:
netflix.isnull().sum()

Show_Id         0
Category        0
Title           0
Director        0
Cast            0
Country         0
Release_Date    0
Rating          0
Duration        0
Type            0
Description     0
dtype: int64

In [27]:
#Duplicate handling
netflix.duplicated().sum()

np.int64(2)

In [29]:
netflix.drop_duplicates(inplace=True) # Dropping of the duplicate column
netflix.duplicated().sum()

np.int64(0)

In [ ]:
#converting the Release date type
netflix["Release_Date"] = netflix["Release_Date"].str.strip() # Handling the white space before and after
netflix["Release_Date"] = pd.to_datetime(netflix["Release_Date"])

In [32]:
netflix.info()

<class 'pandas.DataFrame'>
Index: 7777 entries, 0 to 7788
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Show_Id       7777 non-null   str           
 1   Category      7777 non-null   str           
 2   Title         7777 non-null   str           
 3   Director      7777 non-null   str           
 4   Cast          7777 non-null   str           
 5   Country       7777 non-null   str           
 6   Release_Date  7777 non-null   datetime64[us]
 7   Rating        7777 non-null   str           
 8   Duration      7777 non-null   str           
 9   Type          7777 non-null   str           
 10  Description   7777 non-null   str           
dtypes: datetime64[us](1), str(10)
memory usage: 729.1 KB


In [33]:
# Trimming the unncessary white space
text_columns = [
    "Director",
    "Cast",
    "Country",
    "Type",
    "Title"
]

for column in text_columns:
    netflix[column] = netflix[column].str.strip()

##### Feature Engineering 

##### Overview

Feature engineering is the process of creating new variables or transforming existing ones to improve the quality and interpretability of the dataset. Unlike data cleaning, which focuses on correcting inconsistencies and handling missing values, feature engineering aims to derive meaningful attributes that simplify analysis and reveal additional insights.

The Netflix dataset contains several fields that can be decomposed or transformed into more informative features. For example, the `Release_Date` column can be used to extract temporal attributes such as release year, month, quarter, and weekday, while columns containing multiple comma-separated values, such as `Country` and `Type`, can be used to derive primary categories and count-based features.

These engineered features improve analytical efficiency by reducing repetitive preprocessing during exploratory data analysis and enabling more detailed investigations into temporal trends, content characteristics, and regional distribution.

##### Features to be Engineered

The following features will be created during this stage:

| New Feature | Source Column | Purpose |
|-------------|---------------|---------|
| Release_Year | Release_Date | Analyze yearly content trends |
| Release_Month | Release_Date | Analyze monthly content additions |
| Release_Quarter | Release_Date | Perform quarterly trend analysis |
| Release_Day | Release_Date | Analyze release day patterns |
| Release_Weekday | Release_Date | Identify weekday-wise release behavior |
| Duration_Value | Duration | Extract numerical duration for statistical analysis |
| Duration_Unit | Duration | Distinguish between movie runtime and TV show seasons |
| Primary_Genre | Type | Identify the primary genre of each title |
| Primary_Country | Country | Identify the primary production country |
| Genre_Count | Type | Count the number of genres assigned to each title |
| Country_Count | Country | Count the number of production countries |
| Cast_Count | Cast | Count the number of listed cast members |

These engineered features will serve as the foundation for the subsequent exploratory data analysis and will enable more efficient aggregation, visualization, and business insight generation.

In [34]:
netflix["Release_Year"] = netflix["Release_Date"].dt.year
netflix["Release_Month"] = netflix["Release_Date"].dt.month_name()
netflix["Release_Day"] = netflix["Release_Date"].dt.day
netflix["Release_Quarter"] = netflix["Release_Date"].dt.quarter # Additionally creating a release quarter

In [35]:
netflix["Duration_Value"] = (
    netflix["Duration"]
    .str.extract(r"(\d+)")
    .astype(int)
)

In [39]:
duration = netflix["Duration"].str.extract(r"(\d+)")[0].astype(float)

netflix["Movie_Duration_Minutes"] = duration.where(
    netflix["Category"] == "Movie"
)

netflix["TV_Show_Seasons"] = duration.where(
    netflix["Category"] == "TV Show"
)

In [40]:
netflix[
    [
        "Category",
        "Duration",
        "Movie_Duration_Minutes",
        "TV_Show_Seasons"
    ]
].head(15)

,Category,Duration,Movie_Duration_Minutes,TV_Show_Seasons
0,TV Show,4 Seasons,NaN,4.0
1,Movie,93 min,93.0,NaN
2,Movie,78 min,78.0,NaN
3,Movie,80 min,80.0,NaN
4,Movie,123 min,123.0,NaN
5,TV Show,1 Season,NaN,1.0
6,Movie,95 min,95.0,NaN
7,Movie,119 min,119.0,NaN
8,Movie,118 min,118.0,NaN
9,Movie,143 min,143.0,NaN


In [41]:
netflix["Primary_Genre"] = (
    netflix["Type"]
    .str.split(",")
    .str[0]
    .str.strip()
)

In [42]:
netflix["Primary_Country"] = (
    netflix["Country"]
    .str.split(",")
    .str[0]
    .str.strip()
)

In [43]:
netflix["Genre_Count"] = (
    netflix["Type"]
    .str.split(",")
    .str.len()
)

In [44]:
netflix["Country_Count"] = (
    netflix["Country"]
    .str.split(",")
    .str.len()
)

In [45]:
netflix["Cast_Count"] = (
    netflix["Cast"]
    .str.split(",")
    .str.len()
)

##### Feature Engineering Summary

The feature engineering process successfully enhanced the Netflix dataset by creating several derived attributes from existing variables. Temporal features extracted from the `Release_Date` column provide greater flexibility for analyzing content growth across years, months, quarters, and weekdays. The `Duration` column was separated into numerical values and measurement units, allowing movies and television shows to be analyzed independently.

Additional attributes, including primary genre, primary production country, and count-based features for genres, countries, and cast members, were generated to simplify multi-valued categorical analysis. These transformations reduce the need for repeated preprocessing during exploratory data analysis while improving the dataset's analytical capabilities.

No original information was removed during this process; instead, new variables were derived to enrich the dataset and support more comprehensive univariate, bivariate, and multivariate analyses. The resulting dataset is now better structured for answering the investigation questions and extracting meaningful business insights. The dataset is verfied in the below cell.

In [46]:
# Verifying the data after the feature engineering
netflix.info()

missing_values(netflix)

duplicate_summary(netflix)

netflix.describe(include="all")

<class 'pandas.DataFrame'>
Index: 7777 entries, 0 to 7788
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Show_Id                 7777 non-null   str           
 1   Category                7777 non-null   str           
 2   Title                   7777 non-null   str           
 3   Director                7777 non-null   str           
 4   Cast                    7777 non-null   str           
 5   Country                 7777 non-null   str           
 6   Release_Date            7777 non-null   datetime64[us]
 7   Rating                  7777 non-null   str           
 8   Duration                7777 non-null   str           
 9   Type                    7777 non-null   str           
 10  Description             7777 non-null   str           
 11  Release_Year            7777 non-null   int32         
 12  Release_Month           7777 non-null   str           
 13  Rele

,Show_Id,Category,Title,Director,Cast,Country,Release_Date,Rating,Duration,Type,...,Release_Day,Release_Quarter,Duration_Value,Movie_Duration_Minutes,TV_Show_Seasons,Primary_Genre,Primary_Country,Genre_Count,Country_Count,Cast_Count
count,7777,7777,7777,7777,7777,7777,7777,7777,7777,7777,...,7777.000000,7777.000000,7777.000000,5377.000000,2400.000000,7777,7777,7777.000000,7777.00000,7777.000000
unique,7777,2,7776,4051,6822,682,NaN,14,216,491,...,NaN,NaN,NaN,NaN,NaN,36,82,NaN,NaN,NaN
top,s1,Movie,Consequences,Unknown,Unknown,United States,NaN,TV-MA,1 Season,Documentaries,...,NaN,NaN,NaN,NaN,NaN,Dramas,United States,NaN,NaN,NaN
freq,1,5377,2,2378,718,2549,NaN,2868,1608,334,...,NaN,NaN,NaN,NaN,NaN,1384,2877,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,2019-01-02 19:20:57.708628,NaN,NaN,NaN,...,12.377781,2.598046,69.204706,99.307978,1.760833,NaN,NaN,2.192491,1.22978,7.278385
min,NaN,NaN,NaN,NaN,NaN,NaN,2008-01-01 00:00:00,NaN,NaN,NaN,...,1.000000,1.000000,1.000000,3.000000,1.000000,NaN,NaN,1.000000,1.00000,1.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,2018-02-01 00:00:00,NaN,NaN,NaN,...,1.000000,2.000000,2.000000,86.000000,1.000000,NaN,NaN,2.000000,1.00000,4.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,2019-03-08 00:00:00,NaN,NaN,NaN,...,12.000000,3.000000,88.000000,98.000000,1.000000,NaN,NaN,2.000000,1.00000,8.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-20 00:00:00,NaN,NaN,NaN,...,20.000000,4.000000,106.000000,114.000000,2.000000,NaN,NaN,3.000000,1.00000,10.000000
max,NaN,NaN,NaN,NaN,NaN,NaN,2021-01-16 00:00:00,NaN,NaN,NaN,...,31.000000,4.000000,312.000000,312.000000,16.000000,NaN,NaN,3.000000,12.00000,50.000000


In [47]:
# Saving the dataset for EDA
netflix.to_csv(
    "../data/processed/netflix_cleaned.csv",
    index=False
)

##### Conclusion
The process began with understanding the dataset's structure, identifying the available variables, and evaluating the overall quality of the data. A comprehensive data quality assessment was then performed to examine missing values, duplicate records, data types, and the consistency of the dataset.

Following the assessment, the dataset underwent a series of preprocessing steps, including appropriate data type conversions, standardization of textual data, and cleaning of categorical fields to improve consistency and usability. Feature engineering techniques were subsequently applied to derive additional attributes from existing columns, enriching the dataset with meaningful temporal, categorical, and count-based features that facilitate deeper analysis.

Throughout the preprocessing phase, the integrity of the original data was maintained while enhancing its analytical value. Validation checks were performed after each major transformation to ensure the accuracy and consistency of the processed dataset. Finally, the cleaned and feature-engineered dataset was exported, providing a well-structured and analysis-ready dataset for the subsequent stages of exploratory data analysis, visualization, and business insight generation.